In [ ]:
from dolfinx import log, default_scalar_type
from dolfinx.fem.petsc import NonlinearProblem
import pyvista
import numpy as np
import ufl

from mpi4py import MPI
from dolfinx import fem, mesh, plot

from pathlib import Path
from dolfinx import io

import pickle

log.set_log_level(log.LogLevel.ERROR)

folder = Path("results")
filename = "fenicsBiaxial"

import biaxialGeometry
domain, facet_tags, cell_tags = biaxialGeometry.create_biaxial_geometry(h=3)

In [ ]:
# import pyvista

# cells, types, x = plot.vtk_mesh(domain)
# grid = pyvista.UnstructuredGrid(cells, types, x)
# plotter = pyvista.Plotter()
# plotter.add_mesh(grid, show_edges=True)
# plotter.show()

# Weak form

In [ ]:
import basix.ufl
el_u = basix.ufl.element("Lagrange", domain.basix_cell(), 2, shape=(domain.geometry.dim,))
el_p = basix.ufl.element("Lagrange", domain.basix_cell(), 1)
el_mixed = basix.ufl.mixed_element([el_u, el_p])
W = fem.functionspace(domain, el_mixed)

w = fem.Function(W)

In [ ]:
V_u = W.sub(0) # displacement subspace 
# V_uCollapsed, V_uCollapsed_to_Vu = V_u.collapse()
V_u_x = V_u.sub(0)
V_u_y = V_u.sub(1)
V_u_z = V_u.sub(2)
V_u_xCollapsed, V_u_xCollapsed_to_V_u_x = V_u_x.collapse()
V_u_yCollapsed, V_u_yCollapsed_to_V_u_y = V_u_y.collapse()
V_u_zCollapsed, V_u_zCollapsed_to_V_u_z = V_u_z.collapse()

u_D_left = fem.Function(V_u_xCollapsed)
left_dofs = fem.locate_dofs_topological(V=(V_u_x, V_u_xCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tags.find(1))

u_D_bottom = fem.Function(V_u_xCollapsed)
bottom_dofs = fem.locate_dofs_topological(V=(V_u_y, V_u_yCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tags.find(2))

u_D_front = fem.Function(V_u_zCollapsed)
front_dofs = fem.locate_dofs_topological(V=(V_u_z, V_u_zCollapsed), entity_dim=domain.topology.dim - 1, entities=facet_tags.find(5))


bcs = [fem.dirichletbc(u_D_left, left_dofs, V_u_x), fem.dirichletbc(u_D_bottom, bottom_dofs, V_u_y), fem.dirichletbc(u_D_front, front_dofs, V_u_z)]

In [ ]:
B = fem.Constant(domain, default_scalar_type((0, 0, 0)))
T = fem.Constant(domain, default_scalar_type((0, 0, 0)))

In [ ]:
(u, p) = ufl.split(w)
v_u, v_p = ufl.TestFunctions(W)

In [ ]:
d = len(u) # Spatial dimension
I = ufl.variable(ufl.Identity(d)) # Identity tensor
F = ufl.variable(I + ufl.grad(u)) # Deformation gradient
C = ufl.variable(F.T * F) # Right Cauchy-Green tensor

# Invariants of deformation tensors, see https://en.wikipedia.org/wiki/Invariants_of_tensors
I_1 = ufl.variable(ufl.tr(C))
J = ufl.variable(ufl.det(F))
I_1_bar = ufl.variable(J**(-2/3) * I_1) 
I_2 = 0.5 * (ufl.tr(C) ** 2 - ufl.tr(C * C))

# Material Model

In [ ]:
materialModelUsed = "MooneyRivlinN2" 
materialModelUsed = "MooneyRivlinSimplified" 
materialModelUsed = "NeoHook" 

augmentedLagrangian = True 

if materialModelUsed == "NeoHook": 
    # Neo Hook TPU1
    c_10 = 1.31
    C_10 = fem.Constant(domain, default_scalar_type(c_10))
    psi =  C_10 * (I_1 - 3) + p*(J-1) 

if materialModelUsed == "MooneyRivlinSimplified": 
    # Simplified Mooney Rivlin TPU1
    c_10 = -11.11
    c_01 = 17.4
    c_02 = 3.134

    # Simplified Mooney Rivlin TPU3
    # c_10 = -16.5
    # c_01 = 26.36
    # c_02 = 4.524

    C_10 = fem.Constant(domain, default_scalar_type(c_10))
    C_01 = fem.Constant(domain, default_scalar_type(c_01))
    C_02 = fem.Constant(domain, default_scalar_type(c_02))

    psi = C_10 * (I_1 - 3) + C_01 * (I_2 - 3) + C_02 * (I_2 - 3) ** 2 + p*(J-1)

if materialModelUsed == "MooneyRivlinN2": 
    # Mooney-Rivlin N=2, TPU 1: 
    c_10 = -15.5
    c_01 = 22.49
    c_11 = -0.67
    c_20 = 0.11
    c_02 = 5.47

    C_10 = fem.Constant(domain, default_scalar_type(c_10))
    C_01 = fem.Constant(domain, default_scalar_type(c_01))
    C_11 = fem.Constant(domain, default_scalar_type(c_11))
    C_20 = fem.Constant(domain, default_scalar_type(c_20))
    C_02 = fem.Constant(domain, default_scalar_type(c_02))

    psi = (
    C_10 * (I_1 - 3)
    + C_01 * (I_2 - 3)
    + C_11 * (I_1 - 3) * (I_2 - 3)
    + C_20 * (I_1 - 3)**2
    + C_02 * (I_2 - 3)**2
    + p*(J-1)
    ) 

if augmentedLagrangian: 
    kappa = 10000
    psi += kappa/2*(J-1)**2

P = ufl.diff(psi, F)

sigma = P * 1/J * F.T

In [ ]:
# Define the variational form with traction integral over all facets with value 2. We set the quadrature degree for the integrals to 4.
ds = ufl.Measure("ds", domain=domain, subdomain_data=facet_tags, metadata={"quadrature_degree": 4})
dx = ufl.Measure("dx", domain=domain, metadata={"quadrature_degree": 4})

In [ ]:
sigma_x = fem.Constant(domain, default_scalar_type((0.0, 0.0, 0.0)))
sigma_y = fem.Constant(domain, default_scalar_type((0.0, 0.0, 0.0)))

x_force = 1 # use this to set relative strength of x and y forces. Use parameters of the timestepping loop to set force magnitude. 
y_force = 1

residual = (ufl.inner(ufl.grad(v_u), P)*dx - ufl.inner(v_u, sigma_x)*ds(3) - ufl.inner(v_u, sigma_y)*ds(4) + ufl.inner((J-1), v_p)*dx)


# Solving

In [ ]:
petsc_options = {
    "snes_type": "newtonls",
    "snes_linesearch_type": "none",
    "snes_monitor": None,
    "snes_atol": 1e-8,
    "snes_rtol": 1e-8,
    "snes_stol": 1e-8,
    "ksp_type": "preonly",
    "pc_type": "lu",
    "pc_factor_mat_solver_type": "mumps",
}
problem = NonlinearProblem(
    residual,
    w,
    bcs=bcs,
    petsc_options=petsc_options,
    petsc_options_prefix="hyperelasticity",
)

In [ ]:
log.set_log_level(log.LogLevel.ERROR)

currentFactor = 0
finalFactor = 10 
maxFactorStep = 1 
minFactorStep = 0.05
t = 0 
lastLoopSucceeded = True 
factorStep = maxFactorStep


folder.mkdir(exist_ok=True, parents=True)
xdmf = io.XDMFFile(MPI.COMM_WORLD, folder/f"{filename}.xdmf", "w")
xdmf.write_mesh(domain)
xdmf.write_meshtags(facet_tags, domain.geometry)
xdmf.write_meshtags(cell_tags, domain.geometry)
import pandas as pd 
timestepsTable = []


In [ ]:
def attemptStep(t, factor): 
    # backup current state 
    wSave = fem.Function(W)
    wSave.x.array[:]=w.x.array[:]

    # attempt solving 
    print(f"Attempting time step {t}, factor {factor}")
    sigma_x.value[0] = x_force*factor
    sigma_y.value[1] = y_force*factor
    problem.solve()
    converged = problem.solver.getConvergedReason()
    num_its = problem.solver.getIterationNumber()
    print(f"Solver convergence: {converged}. Number of iterations {num_its}")
    if converged < 0: 
        print(f"Solver did not converge on time step {t}, factor {factor}")
        # reset to backup 
        w.x.array[:] = wSave.x.array[:]
        return False 
    else: 
        return True 

In [ ]:
def writeResults():
    # write results and all that 
    # u 
    V_u_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,)))
    u_out = fem.Function(V_u_out)
    u_out.name = "u"
    u_out.interpolate(w.sub(0).collapse())
    xdmf.write_function(u_out, t)

    # p
    V_p_out = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1))
    p_out = fem.Function(V_p_out)
    p_out.name = "p"
    p_out.interpolate(w.sub(1).collapse())
    xdmf.write_function(p_out, t)

    # J
    V_J_post = fem.functionspace(domain, ("Lagrange", 1))
    J_post = fem.Expression(J, V_J_post.element.interpolation_points)
    J_out = fem.Function(V_J_post)
    J_out.name = "J"
    J_out.interpolate(J_post)
    xdmf.write_function(J_out, t)

    # F
    V_F_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    F_post = fem.Expression(F, V_F_post.element.interpolation_points)
    F_out = fem.Function(V_F_post)
    F_out.name = "F"
    F_out.interpolate(F_post)
    xdmf.write_function(F_out, t)

    # P 
    V_P_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    P_post = fem.Expression(P, V_P_post.element.interpolation_points)
    P_out = fem.Function(V_P_post)
    P_out.name = "P"
    P_out.interpolate(P_post)
    xdmf.write_function(P_out, t)

    # sigma 
    V_sigma_post = fem.functionspace(domain, basix.ufl.element("P", domain.basix_cell(), 1, shape=(3,3,)))
    sigma_post = fem.Expression(sigma, V_sigma_post.element.interpolation_points)
    sigma_out = fem.Function(V_sigma_post)
    sigma_out.name = "sigma"
    sigma_out.interpolate(sigma_post)
    xdmf.write_function(sigma_out, t)

    # into dataframe
    timestepsTable.append([x_force*currentFactor, y_force*currentFactor])

In [ ]:
# init with factor 0 
attemptStep(t=0, factor=0)
writeResults()
t +=1 

while currentFactor < finalFactor: 

    if lastLoopSucceeded: 
        factorStep = min(factorStep*2, maxFactorStep)
    else: 
        factorStep = factorStep / 2 

    if factorStep < minFactorStep: 
        print(f"Aborted at {currentFactor} factor because stepsize got too small.")
        break 

    success = attemptStep(t=t, factor=currentFactor + factorStep)

    if success: 
        lastLoopSucceeded = True 
        currentFactor = currentFactor + factorStep
        t += 1 
        writeResults()

    else: 
        lastLoopSucceeded = False 


In [ ]:
timestepsDf = pd.DataFrame(columns=["X_force", "Y_force"], data=timestepsTable)

with open(f"{folder}/{filename}TimeSteps.pickle", 'wb') as f:
    # Pickle the 'data' dictionary using the highest protocol available.
    pickle.dump(timestepsDf, f, pickle.HIGHEST_PROTOCOL)

xdmf.close()

In [ ]:
timestepsDf